### 一.一切的开始-数据的转化


GPT等模型是基于transformer架构搭建的，transformer架构处理数据的方法是将离散的数据(单词，句子，段落，音频，视频)转换为连续空间中的点(多维空间)，即转化为向量形式，这个过程称为嵌入，不同的数据格式一般使用的是不同的嵌入模型，在文本类型的数据中，词嵌入是最广为使用的，但是句子和段落嵌入检索增强生成(RAG)领域也广为流行，词嵌入中最常见的是word2vec方法。

一般来说越细微，越高级的模型，嵌入向量的维度会越高如GPT-3的每一个向量就有12228维，这使只能看到三维空间的人类很难可视化认知(这里推荐观看油管上3blue的关于transformer的视频)，但是高维度的代价就是运算速度的降低

### 二.文本分词

我们首先要将文本分割为独立的词元(即token)，词元可以是单词，也可以是标点符号等，下面拿一篇小说来进行词元分割

In [2]:
from torch.nn.functional import embedding

with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()#利用python的读取文件操作，将小说内容用raw_text实例化
print("total number of the text is:",len(raw_text))
print(raw_text[:99])#打印出前九十九个字节,注意这里不是单词数

total number of the text is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


非常好，我们成功的读取了文本，下面将使用正则库re分割文本

In [11]:
import re
text="Hello world.This is a test"
result=re.split(r'(\s)',text)#将这一个句子按空格分割
print(result)

['Hello', ' ', 'world.This', ' ', 'is', ' ', 'a', ' ', 'test']


我们尽量不要将文本统一为小写字母，因为大写字母有时会有特殊的含义，下面再按照逗号句号分割

In [10]:
result=re.split(r'([, .]|\s)',text)
print(result)

['Hello', ' ', 'world', '.', 'This', ' ', 'is', ' ', 'a', ' ', 'test']


上述代码中仍然有的是空白字符，下面安全的删掉它们

In [9]:
result=[item for item in result if item.strip()]
print(result)

['Hello', 'world', '.', 'This', 'is', 'a', 'test']


niceeeeeee!!!!直接实现了，但是空白字符的删除与否与实际问题相关，如在python相关的问题中，由于python对缩进极其敏感，所以可以适时保留空白符，当然，如果去掉了，运行速度会更快，下面我们构建一个有更多标点符号的分词器，来对小说的前一百个单词分词

In [8]:

preprocessed = re.split(r'([,.:;?_!"()\']|\s)', raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4506
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--though', 'a', 'good', 'fellow', 'enough--so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his']


### 三.将词元转换为词元ID

我们需要将词元转换为整数，即词元ID，所以必须用到一张词汇表，每个单词对应的是一个整数，这个表的实现过程很简单：首先将文本分成独立的词元，将这些词元按字母顺序排序，删除重复的词元，将唯一的词元聚合到一张词汇表中，接下来试一下

In [12]:
all_word=sorted(set(preprocessed))
vocab_size=len(all_word)
print(vocab_size)#下面实现词汇表
vocab={token:integer for integer,token in enumerate(all_word)}
for i,item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break

1200
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--and', 6)
('--even', 7)
('--it', 8)
('--oh', 9)
('--she', 10)
('--that', 11)
('.', 12)
(':', 13)
(';', 14)
('?', 15)
('A', 16)
('Ah', 17)
('Ah--I', 18)
('Among', 19)
('And', 20)
('Are', 21)
('Arrt', 22)
('As', 23)
('At', 24)
('Be', 25)
('Begin', 26)
('Burlington', 27)
('But', 28)
('By', 29)
('Carlo', 30)
('Chicago', 31)
('Claude', 32)
('Come', 33)
('Croft', 34)
('Destroyed', 35)
('Devonshire', 36)
('Don', 37)
('Dubarry', 38)
('Emperors', 39)
('Florence', 40)
('For', 41)
('Gallery', 42)
('Gideon', 43)
('Gisburn', 44)
('Gisburn--as', 45)
('Gisburn--fond', 46)
('Gisburns', 47)
('Grafton', 48)
('Greek', 49)
('Grindle', 50)


至此，我们已经实现了将单词对应为整数的单词表，不包括空格和标点符号，这个表是根据训练集得到的，但是不止适用于训练集，同时，为了将数字对应回文本，我们还需要一张逆向词汇表，所以完整的分词器类，需要有encode和decode方法，实现文本到整数再到文本的转化

In [38]:
import re

class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}#创建逆向词汇表，将词元ID映射回原始文本，这是利用了python的字典特性，将键值对换

    def encode(self,text):
        preprocessed=re.split(r'([,.?_!"()\'\\]|\s)',text) # 修正了正则，去掉了字符类里的 ||
        preprocessed=[item for item in preprocessed if item.strip()]
        ids=[self.str_to_int.get(s, 0) for s in preprocessed] # 使用 get 方法防止 KeyError，找不到时默认返回 0
        return ids

    def decode(self,ids):#将词元ID转换为文本
        text=" ".join([self.int_to_str[i] for i in ids])#join方法将单独的字符拼接成句子
        text=re.sub(r'\s([,.?_!"()\'\\])', r'\1', text) # 修正了正则，真正移除标点符号前的空格，且保留了感叹号 !
        return text

tokenizer=SimpleTokenizerV1(vocab)
text="""it is the last he plained,you know,Mrs.Gisburn said with pardonable pride"""
ids=tokenizer.encode(text)
print(ids)

[619, 618, 1053, 640, 559, 0, 5, 1195, 634, 5, 77, 12, 44, 911, 1177, 809, 853]


In [32]:
print(tokenizer.decode(ids))

it is the last he!, you know, Mrs. Gisburn said with pardonable pride


可以实现简单的基础分词功能，但当输入文本中出现了分词器中没有的单词，就会报错，所以，一般使用规模更大且更多样化的训练集来扩展词汇表

### 四.引入特殊上下文词元

为了处理没有出现过的单词，我们要对simpletokenizer改进，使其可以处理未知的词汇，为此，我们引入了<[unk]>表示未出现在训练集中的单词，而<[endotext]>用来分隔开两个不相关的文本来源,虽然计算机在训练时是将文本连续处理的，但是实际上是相互独立的，有助于提高模型的理解能力

In [40]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)
#如代码所示，比修改之前多了两个词元，下面实现完整版

1202
('younger', 1197)
('your', 1198)
('yourself', 1199)
('<|endoftext|>', 1200)
('<|unk|>', 1201)


In [45]:
import re

class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={s:i for i,s in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.?_!"()\'\\]|\s)',text)
        preprocessed=[item for item in preprocessed if item.strip()]
        preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids=[self.str_to_int[i] for i in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        text=re.sub(r'\s([,.?_!"()\'\\])', r'\1', text)
        return text

text1="Hello,do you like tea?"
text2="In the sunlit terraces of the palace"
text="<|endofest|> ".join([text1, text2])#小试牛刀
print(text)
tokenizer=SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

Hello,do you like tea?<|endofest|> In the sunlit terraces of the palace
[1201, 5, 377, 1195, 667, 1040, 15, 1201, 63, 1053, 1019, 1049, 770, 1053, 1201]
<|unk|>, do you like tea? <|unk|> In the sunlit terraces of the <|unk|>


要注意，一般会引入的特殊词元是[BOS]序列开始,[EOS]序列结束,[PAD]填充,当训练批次不为一时，可以通过填充使较短的词元进行扩展或填充，来匹配最长文本的长度,要注意GPT使用的分词器比那个不依赖于这些特殊词元，只是用<endofest>,也不使用<|unk|>

### 五.BPE

BPE是一种更复杂的分词方案，因此使用python开源的tiktoken，基于Rust的源代码实现了BPE

In [2]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
text=("Hello,do you like tea?<endofest>In the sunlit terraces of the palace""of someunknownPlace.")
integer=tokenizer.encode(text,allowed_special={"<|endofest|>"})
print(integer)

[15496, 11, 4598, 345, 588, 8887, 30, 27, 437, 1659, 395, 29, 818, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 1659, 617, 34680, 27271, 13]


In [3]:
strings=tokenizer.decode(integer)
print(strings)

Hello,do you like tea?<endofest>In the sunlit terraces of the palaceof someunknownPlace.


以上有两个特点值得注意，第一个就是endofest被分配了一个较大的词元ID，第二，BPE甚至不需要unk来处理未知词汇，事实上，原理是将未知的词元分解成更小的词元甚至是单个字符，从而可以处理不熟悉的单词，然后BPE可以将频繁出现的字符合并为子词，比如将d,e存入词汇表，这是define，hidden，depend中的常见组合，字符和子词的合并由一个频率阈值决定

### 六.使用滑动窗口进行数据采样

为了生成嵌入向量，接下来要生成用于训练模型的输入-目标对，通过输入输出对，可以将一段数据反复训练，多次使用，不断根据前文预测后文，再和正确的标签比对，这里的关键是不断向前，但是不能让后文影响前文，我们使用滑动窗口来做到

In [7]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()
enc_text=tokenizer.encode(raw_text)
print(len(enc_text))#首先使用分词器进行全文分词
enc_sample=enc_text[50:]


5145


创建下一单词预测任务的输入-目标对的方法很简单：定义两个变量x，y，变量x储存输入的词元，y则用于储存由x的每个输入词元右移一个位置所得的目标词元

In [8]:
context_size=4#每次输入四个词元
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]#刚好差了一个，在x拿到前三个词元时，需要预测第四个，而y就是标签
print(f"x:{x},y:{y}")

x:[290, 4920, 2241, 287],y:[4920, 2241, 287, 257]


In [10]:
for i in range(1,context_size+1):
    context=enc_sample[:i]#开头到第i-1个，不含i
    desired=enc_sample[i]
    print(context,"---->",desired)#箭头左侧表示的是大模型接受的输入，右侧表示大模型应该预测的目标词元的ID
for i in range(1,context_size+1):
    context=enc_sample[:i]#开头到第i-1个，不含i
    desired=enc_sample[i]
    print(tokenizer.decode(context),"---->",tokenizer.decode([desired]))#反向将词元ID转换为文本

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


ok，现在输入输出对已经创建好了，距离将词元转换为嵌入向量前，只需要再实现一个数据加载器，让其遍历数据集，将输入和目标以pytorch张量的形式返回，我们使用pytorch自带的Dataset和Dataloader类完成

In [13]:
import torch
from torch.utils.data import DataLoader,Dataset
class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):#注意stride要小于max
        self.input_ids=[]
        self.target_ids=[]#创建空列表储存输入输出对
        token_ids=tokenizer.encode(txt)#对全部文本进行分词
        for i in range(0,len(token_ids)-max_length,stride):#使用滑动窗口将文本划分为长度为max_length的重叠序列
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)
    def __len__(self):#返回数据集总行数
        return len(self.input_ids)
    def __getitem__(self, idx):#返回数据集的指定行
        return self.input_ids[idx],self.target_ids[idx]


下面借助dataloader

In [42]:
def create_dataloader_v1(txt, batch_size=4, max_len=512, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_len, stride)

    # 修复版拼接函数：先将列表转为张量，再进行堆叠
    def custom_collate(batch):
        # 1. 先将 batch 中的 inputs 和 targets 分开
        # 此时 inputs 和 targets 是由 Python 列表组成的元组
        inputs, targets = zip(*batch)

        # 2. 关键步骤：将 Python 列表显式地转换为 PyTorch 张量
        # 这一步告诉 PyTorch："把这些数字列表变成我可以计算的张量"
        inputs = [torch.tensor(item) for item in inputs]
        targets = [torch.tensor(item) for item in targets]

        # 3. 现在可以安全地堆叠了
        return torch.stack(inputs), torch.stack(targets)

    dataloader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=shuffle,
                            drop_last=drop_last,
                            num_workers=num_workers,
                            collate_fn=custom_collate)
    return dataloader

现在数据集已经创建好了，读取器也已经就绪了，直接使用一下

In [43]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()
dataloader=create_dataloader_v1(raw_text,batch_size=1,max_len=4,stride=1,shuffle=False)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

(tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]]))


### 七.创建词元嵌入

在创建词汇表的过程中，实际上我们的词汇表是具有嵌入向量的，每一行代表的都是一个单词，而每一列代表的是一个维度，里面的值都是随机的，也是在训练过程中不断被优化的，当我们将ID化后的tensor传入嵌入层时，实际上是在权重矩阵中检索出相应的行，比如你传入ID为3的tensor，就去取出第三行的所有参数

In [18]:
input_ids=torch.tensor([2,3,5])
vocab_size=6
output_dim=3
torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
print(embedding_layer.weight)
#打印出权重参数看看
print(embedding_layer(torch.tensor([3])))#看，实际上就是取出来表中第三行
print(embedding_layer(input_ids))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096]], grad_fn=<EmbeddingBackward0>)


### 八.编码单词位置信息

transformer有一个重大的缺陷，无法感知ID在输入序列的位置，因为相同的ID总被映射到相同的向量，导致出现在不同位置的相同ID,即使拥有不同含义，却不能被识别出来，虽然自注意力机制本身与位置无关，但毫无疑问注入额外的位置信息是有帮助的。这里有两种策略
：

1.绝对位置嵌入：向对应次元的嵌入向量中添加一个独特的位置嵌入，明确确切位置

2：相对位置嵌入：关注词元之间的相对位置或者距离，这种方法能更好的适应不同长度的序列


OpenAI的GPT模型用的是绝对位置嵌入，这些嵌入会在模型训练的过程中被优化

In [49]:
vocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
max_length=4
dataloader=create_dataloader_v1(raw_text,batch_size=8,max_len=4,stride=4,shuffle=False,drop_last=True,num_workers=0)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)
print(inputs)
print("\nInputs shape:\n",inputs.shape)


tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


如上所示，ID张量的维度为8*4,表示包含八个文本样本，每个文本由四个词元构成，现在使用嵌入层将这些词元ID嵌入256维的向量中：

In [50]:
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


可见，每一个词元ID都已被嵌入一个256维的向量中，为实现GPT模型的绝对位置嵌入，只需要创建一个维度与token_embedding_layer相同的嵌入层即可：

In [51]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)
pos_embedding_layer=pos_embedding_layer(torch.arange(context_length))
print(pos_embedding_layer.shape)

torch.Size([4, 256])


pos_embedding的输入通常是一个占位符向量torch.arange(context_length)，它包含一个从零开始递增，直至最大输入长度减一的数值序列，context_length是一个变量，表示模型支持的输入块的最大长度，我们将其设置为与输入文本的最大长度一致，但是实际中可能会超过模型支持的块大小，这时候需要截断文本

pytorch会在每个批次中的每个4*256维的词元嵌入向量中添加一个pos_embedding张量

In [52]:
input_embeddings=token_embeddings+pos_embedding_layer
print(input_embeddings.shape)

torch.Size([8, 4, 256])


这样我们就实现了在transformer作用前的数据处理一共有以下几步:1.输入文本--->2.词元化文本--->3.转化为词元ID--->4.转化为词元嵌入--->5.加上位置嵌入--->得到输入嵌入，接下来就输入解码器了